# Remote Inference — Cognitive Similarity

Runs TRIBE v2 inference on IBC stimuli and saves raw tensors to Google Drive.
All cells are safe to re-run — already-processed stimuli are skipped automatically.

**Runtime:** Google Colab Pro with A100 GPU.

## Cell 1 — Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone/pull the git repo
import os
REPO_URL = "https://github.com/YOUR_USERNAME/cognitive-similarity.git"  # placeholder
REPO_DIR = "/content/cognitive-similarity"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Install dependencies
!pip install -q tribev2 nilearn scikit-learn hypothesis

# HuggingFace login (for gated Llama 3.2 access used by TRIBE v2)
from huggingface_hub import login
login()  # will prompt for token

## Cell 2 — Load Models

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from tribev2 import TribeModel

CACHE_FOLDER = "/content/drive/MyDrive/cognitive-similarity-cache"

# Cortical model (default)
cortical_model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE_FOLDER)

# Subcortical model
subcortical_model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={
        "data.neuro.projection": {
            "name": "MaskProjector",
            "mask": "subcortical",
            "=replace=": True
        },
        "data.neuro.fwhm": 6.0
    }
)

print("Models loaded successfully.")

## Cell 3 — Download IBC Stimuli and Build Manifest

In [ ]:
import json
import requests
from pathlib import Path
from cognitive_similarity.cache import ResponseCache

DRIVE_CACHE = Path(CACHE_FOLDER)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

# IBC FaceBody stimuli base URL
IBC_BASE = "https://raw.githubusercontent.com/individual-brain-charting/public_protocols/master/FaceBody/stimuli"

# Define stimulus manifest entries
# Each entry: stimulus_id, category, modality, github_url
STIMULUS_DEFINITIONS = [
    # Visual System stimuli (faces, places, bodies, written characters)
    {"stimulus_id": "face_01", "category": "face", "modality": "image", "github_url": f"{IBC_BASE}/adult/image_001.jpg"},
    {"stimulus_id": "face_02", "category": "face", "modality": "image", "github_url": f"{IBC_BASE}/adult/image_002.jpg"},
    {"stimulus_id": "place_01", "category": "place", "modality": "image", "github_url": f"{IBC_BASE}/places/image_001.jpg"},
    {"stimulus_id": "place_02", "category": "place", "modality": "image", "github_url": f"{IBC_BASE}/places/image_002.jpg"},
    {"stimulus_id": "body_01", "category": "body", "modality": "image", "github_url": f"{IBC_BASE}/body/image_001.jpg"},
    {"stimulus_id": "body_02", "category": "body", "modality": "image", "github_url": f"{IBC_BASE}/body/image_002.jpg"},
    # Add more as needed — this is a representative subset
]

# Download stimuli and compute content hashes
cache = ResponseCache(str(DRIVE_CACHE))
manifest = []

for defn in STIMULUS_DEFINITIONS:
    local_path = DRIVE_CACHE / "stimuli" / defn["stimulus_id"]
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        print(f"Downloading {defn['stimulus_id']}...")
        r = requests.get(defn["github_url"])
        r.raise_for_status()
        local_path.write_bytes(r.content)

    from cognitive_similarity.models import Stimulus
    stim = Stimulus(video_path=str(local_path), stimulus_id=defn["stimulus_id"])
    content_hash = cache._content_hash(stim)

    manifest.append({
        **defn,
        "local_path": str(local_path),
        "content_hash": content_hash,
        "tensor_dir": f"tensors/{content_hash}",
    })

manifest_path = DRIVE_CACHE / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Manifest written: {len(manifest)} stimuli")

## Cell 4 — Resume-from-Checkpoint Inference Loop

In [ ]:
from cognitive_similarity.stimulus_runner import StimulusRunner
from cognitive_similarity.models import Stimulus
import numpy as np

runner = StimulusRunner(cortical_model, subcortical_model)

processed = skipped = 0

for entry in manifest:
    stim = Stimulus(video_path=entry["local_path"], stimulus_id=entry["stimulus_id"])
    tensor_dir = DRIVE_CACHE / entry["tensor_dir"]
    cortical_path = tensor_dir / "raw_cortical.npy"

    if cortical_path.exists():
        skipped += 1
        continue

    print(f"Processing {entry['stimulus_id']}...")
    brain_response = runner.run(stim)

    tensor_dir.mkdir(parents=True, exist_ok=True)
    np.save(cortical_path, brain_response.cortical)
    np.save(tensor_dir / "raw_subcortical.npy", brain_response.subcortical)
    processed += 1

print(f"Done. Processed: {processed}, Skipped (cached): {skipped}, Total: {len(manifest)}")

## Cell 5 — Verify Cache

In [ ]:
import numpy as np
from pathlib import Path

print("=== Cache Verification ===\n")
tensors_dir = DRIVE_CACHE / "tensors"

for entry in manifest:
    tensor_dir = tensors_dir / entry["content_hash"]
    cortical_path = tensor_dir / "raw_cortical.npy"
    subcortical_path = tensor_dir / "raw_subcortical.npy"

    if cortical_path.exists():
        cortical = np.load(cortical_path)
        subcortical = np.load(subcortical_path) if subcortical_path.exists() else None
        sub_shape = subcortical.shape if subcortical is not None else "missing"
        print(f"\u2713 {entry['stimulus_id']:30s}  cortical={cortical.shape}  subcortical={sub_shape}")
    else:
        print(f"\u2717 {entry['stimulus_id']:30s}  NOT CACHED")